# Question 8 — Quelle politique par type de séjour, et qui remplit la semaine ?

**Auteur : Wenceslas Bouity** — Projet Valmont, Datamining B3 (ECE Paris), sujet 9.

La question 7 a montré que Valmont a **quatre types de séjours** et non les trois segments déclarés. Cette
question en tire les conséquences commerciales : pour chaque type, qu'est-ce qu'il rapporte, qu'est-ce qu'il
coûte en annulations, quand remplit-il l'hôtel, et quelle condition de vente faut-il lui appliquer ?

Elle répond aussi à une question précise de la direction : **qui remplit les nuits de semaine ?**

**Méthode** : profil chiffré de chaque type (prix, annulation, services, part en semaine), décomposition du
revenu perdu, puis une décision par type avec son gain estimé et sa plage de sensibilité.

**Point de départ** : `data/clean/valmont_clean.csv` et `data/clean/q7_types_sejours.csv`, le fichier de types
produit par la question 7.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from scipy.stats import chi2_contingency

pd.set_option("display.width", 210)
pd.set_option("display.max_columns", 40)

# Memes couleurs qu'en Q7, attachees au type et non a son rang :
# un type garde sa couleur d'un graphique a l'autre et d'un notebook a l'autre.
COULEUR_TYPE = {
    "Court séjour de dernière minute": "#00b894",
    "Long séjour individuel": "#f2a600",
    "Groupe et séminaire": "#257de0",
    "Réservation très anticipée": "#ec4647",
}
# Ordre de lecture : du type le plus sur au plus risque (taux d'annulation croissant, Q7).
ORDRE = list(COULEUR_TYPE)
# Etiquettes courtes, pour les axes des graphiques seulement.
COURT = {"Court séjour de dernière minute": "Court séjour\ndernière minute",
         "Long séjour individuel": "Long séjour\nindividuel",
         "Groupe et séminaire": "Groupe et\nséminaire",
         "Réservation très anticipée": "Réservation\ntrès anticipée"}
ENCRE, ENCRE_2, GRILLE = "#191919", "#666666", "#dcdcdc"
plt.rcParams.update({
    "figure.dpi": 110, "figure.facecolor": "white", "axes.facecolor": "white",
    "axes.edgecolor": GRILLE, "axes.labelcolor": ENCRE, "axes.titlesize": 11,
    "axes.titleweight": "bold", "axes.titlecolor": ENCRE, "axes.grid": True,
    "grid.color": GRILLE, "grid.linewidth": 0.6, "axes.axisbelow": True,
    "xtick.color": ENCRE_2, "ytick.color": ENCRE_2, "font.size": 9,
    "axes.spines.top": False, "axes.spines.right": False, "legend.frameon": False,
})

reservations = pd.read_csv("../data/clean/valmont_clean.csv")
types_q7 = pd.read_csv("../data/clean/q7_types_sejours.csv")

# Jointure interne : les 6 anomalies de la Q9 n'ont pas de type et sortent ici,
# exactement comme elles etaient sorties de la classification en Q7.
sejours = reservations.merge(types_q7, on="reservation_id", how="inner")

print("Réservations du fichier propre :", len(reservations))
print("Réservations typées en Q7      :", len(types_q7))
print("Après jointure                 :", len(sejours))
print("Perdues dans la jointure       :", len(types_q7) - len(sejours))
print("\nEffectif par type :")
print(sejours["type_sejour"].value_counts().reindex(ORDRE).to_string())

Réservations du fichier propre : 800
Réservations typées en Q7      : 794
Après jointure                 : 794
Perdues dans la jointure       : 0

Effectif par type :
type_sejour
Court séjour de dernière minute    363
Long séjour individuel             171
Groupe et séminaire                134
Réservation très anticipée         126


## 1. Contrôle de la jointure

La jointure est faite sur `reservation_id`, identifiant unique vérifié en question 7. Les 6 réservations
marquées comme anomalies n'ont pas de type : elles sont écartées ici comme elles l'étaient de la
classification, et restent traitées en question 9. Aucune ligne n'est perdue autrement, et les effectifs par
type sont identiques à ceux de la question 7 — c'est ce que contrôle la cellule ci-dessus.

In [2]:
# Deux grandeurs de revenu servent dans tout le notebook, definies une fois ici.
# revenu_potentiel : ce que la reservation aurait rapporte en chambre si elle avait ete honoree.
# Le prix est un prix par nuit et par chambre (verifie dans le notebook 00), d'ou le produit.
sejours["revenu_potentiel"] = sejours["prix_nuit_eur"] * sejours["nb_nuits"]
sejours["revenu_realise"] = sejours["revenu_potentiel"] * (1 - sejours["a_annule"])
sejours["revenu_perdu"] = sejours["revenu_potentiel"] * sejours["a_annule"]
sejours["petit_dejeuner_oui"] = (sejours["petit_dejeuner"] == "oui").astype(int)
sejours["en_semaine"] = 1 - sejours["weekend"]

print("Sur les", len(sejours), "réservations analysées :")
print(f"  revenu potentiel : {sejours['revenu_potentiel'].sum():>10,.0f} €")
print(f"  revenu réalisé   : {sejours['revenu_realise'].sum():>10,.0f} €")
print(f"  revenu annulé    : {sejours['revenu_perdu'].sum():>10,.0f} €  "
      f"({sejours['revenu_perdu'].sum() / sejours['revenu_potentiel'].sum():.1%} du potentiel)")
print(f"\n  nuitées vendues  : {sejours.loc[sejours['a_annule'] == 0, 'nb_nuits'].sum():>6,.0f}")
print(f"  nuitées annulées : {sejours.loc[sejours['a_annule'] == 1, 'nb_nuits'].sum():>6,.0f}")

Sur les 794 réservations analysées :
  revenu potentiel :    454,228 €
  revenu réalisé   :    338,948 €
  revenu annulé    :    115,279 €  (25.4% du potentiel)

  nuitées vendues  :  1,720
  nuitées annulées :    620
